# Example 14 — Beamforming with a ULA

**Level:** Intermediate

After working through this notebook you will know how to:

- Apply **Delay-and-Sum** (DAS) conventional beamforming
- Apply **MVDR** (Capon) minimum-variance distortionless-response beamforming
- Apply **LCMV** beamforming with simultaneous gain and null constraints
- Visualise and compare normalised beam patterns
- Use `compute_beam_pattern()` to evaluate spatial filter response

In [ ]:
import sys

sys.path.insert(0, ".")

import numpy as np

%matplotlib inline
import matplotlib.pyplot as plt
from plot_helpers import savefig
from spectra.algorithms import compute_beam_pattern, delay_and_sum, lcmv, mvdr
from spectra.algorithms.beamforming import _mvdr_weights
from spectra.arrays import ula
from spectra.datasets import DirectionFindingDataset
from spectra.waveforms import QPSK

FREQ_HZ     = 2.4e9
N_ELEMENTS  = 8
SPACING     = 0.5
N_SNAPSHOTS = 512
SAMPLE_RATE = 1e6
SEED        = 0

SCAN_DEG = np.linspace(1, 179, 512)
SCAN_RAD = np.deg2rad(SCAN_DEG)

## 1. Array and Dataset Setup

We use the same `DirectionFindingDataset` from example 13, but configure it
with two sources: a desired signal and a strong interferer. The spatial mixing
inside the dataset produces a multi-element snapshot matrix from which we
can extract `X = I + jQ` for each element.

In [ ]:
arr = ula(num_elements=N_ELEMENTS, spacing=SPACING, frequency=FREQ_HZ)

ds = DirectionFindingDataset(
    array=arr,
    signal_pool=[QPSK(samples_per_symbol=4)],
    num_signals=2,
    num_snapshots=N_SNAPSHOTS,
    sample_rate=SAMPLE_RATE,
    snr_range=(10.0, 20.0),
    azimuth_range=(np.deg2rad(10), np.deg2rad(170)),
    elevation_range=(0.0, 0.0),
    min_angular_separation=np.deg2rad(30),
    num_samples=50,
    seed=SEED,
)

data, target = ds[0]
X = data[:, 0, :].numpy() + 1j * data[:, 1, :].numpy()

print(f"Sources: az = {np.rad2deg(target.azimuths)} degrees")
print(f"SNRs:    {target.snrs} dB")

az_desired    = float(target.azimuths[0])
az_interferer = float(target.azimuths[1])

## 2. Delay-and-Sum

The simplest beamformer: steer toward the target by applying conjugate steering-vector
weights, then average across elements. No covariance estimate needed — works even
with a single snapshot. The weight vector is:

**w** = **a**(θ)* / N

The main limitation is that DAS does not null interference — it suppresses
other directions only through the natural sidelobe structure of the array.

In [ ]:
y_das = delay_and_sum(X, arr, target_az=az_desired)
print(f"DAS output power: {10*np.log10(np.mean(np.abs(y_das)**2)):.1f} dB")

freqs = np.fft.fftshift(np.fft.fftfreq(N_SNAPSHOTS, d=1.0/SAMPLE_RATE))
psd = np.abs(np.fft.fftshift(np.fft.fft(y_das)))**2

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(freqs/1e3, 10*np.log10(psd + 1e-30), linewidth=0.8)
ax.set_xlabel("Frequency (kHz)")
ax.set_ylabel("Power (dB)")
ax.set_title("DAS Output Spectrum")
ax.grid(True, alpha=0.3)
plt.tight_layout()
savefig("14_das_spectrum.png")
plt.show()

## 3. MVDR Beamformer

MVDR (Minimum Variance Distortionless Response), also known as the Capon beamformer,
minimises output power while maintaining a unit-gain constraint at the target direction:

minimise **w**ᴴ **R** **w**  subject to  **w**ᴴ **a**(θ) = 1

Solution: **w** = **R**⁻¹ **a** / (**a**ᴴ **R**⁻¹ **a**)

MVDR uses the sample covariance **R** = **X** **X**ᴴ / T to adapt to the
interference environment, automatically placing nulls toward interferers.

In [ ]:
y_mvdr = mvdr(X, arr, target_az=az_desired)
print(f"MVDR output power: {10*np.log10(np.mean(np.abs(y_mvdr)**2)):.1f} dB")

# Verify distortionless constraint
w_mvdr = _mvdr_weights(X, arr, target_az=az_desired)
a = arr.steering_vector(azimuth=az_desired, elevation=0.0)
print(f"|w^H a| = {abs(w_mvdr.conj() @ a):.4f}  (should be ≈ 1.0)")

## 4. LCMV with Null Steering

LCMV (Linearly Constrained Minimum Variance) generalises MVDR to multiple
simultaneous linear constraints:

minimise **w**ᴴ **R** **w**  subject to  **C**ᴴ **w** = **g**

where **C** = [**a**(θ₁), **a**(θ₂), ...] and **g** = [g₁, g₂, ...].

Setting g₁ = 1 (desired signal) and g₂ = 0 (interferer null) forces the
beamformer to place an exact null in the interference direction regardless
of its power.

In [ ]:
y_lcmv = lcmv(
    X, arr,
    constraints=[(az_desired, 0.0), (az_interferer, 0.0)],
    responses=[1.0 + 0j, 0.0 + 0j],
)
print(f"LCMV output power: {10*np.log10(np.mean(np.abs(y_lcmv)**2)):.1f} dB")

# Verify null depth at interferer direction
w_lcmv = lcmv(
    X, arr,
    constraints=[(az_desired, 0.0), (az_interferer, 0.0)],
    responses=[1.0 + 0j, 0.0 + 0j],
    return_weights=True,
)
a_int = arr.steering_vector(azimuth=az_interferer, elevation=0.0)
print(f"|w^H a_interferer| = {abs(w_lcmv.conj() @ a_int):.6f}  (should be ≈ 0.0)")

## 5. Beam Pattern Comparison

`compute_beam_pattern()` sweeps a fixed weight vector across all scan angles
and returns the normalised power response. This lets us see exactly where each
beamformer places its main lobe, sidelobes, and nulls.

In [ ]:
w_das = arr.steering_vector(azimuth=az_desired, elevation=0.0).conj() / N_ELEMENTS

fig, ax = plt.subplots(figsize=(9, 4))
for name, w, color in [
    ("DAS",  w_das,  "steelblue"),
    ("MVDR", w_mvdr, "seagreen"),
    ("LCMV", w_lcmv, "darkorange"),
]:
    pattern = compute_beam_pattern(w, arr, SCAN_RAD)
    ax.plot(SCAN_DEG, 10 * np.log10(pattern + 1e-12), label=name, color=color, linewidth=1.3)

ax.axvline(np.rad2deg(az_desired),    color="crimson", linestyle="--", linewidth=1.2,
           label=f"Desired {np.rad2deg(az_desired):.0f}°")
ax.axvline(np.rad2deg(az_interferer), color="black",   linestyle=":",  linewidth=1.2,
           label=f"Interferer {np.rad2deg(az_interferer):.0f}°")
ax.set_xlabel("Azimuth (degrees)")
ax.set_ylabel("Normalised beam pattern (dB)")
ax.set_title(f"Beam Pattern Comparison — {N_ELEMENTS}-element ULA")
ax.set_ylim(-50, 5)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
savefig("14_beam_patterns.png")
plt.show()